In [1]:
from argparse import ArgumentParser
from pathlib import Path

from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import LocalStore
from virtualizarr import open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from dask.distributed import Client

client = Client(threads_per_worker=1)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 48
Total threads: 48,Total memory: 188.56 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38671,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38275,Total threads: 1
Dashboard: /proxy/44705/status,Memory: 3.93 GiB
Nanny: tcp://127.0.0.1:41401,


In [2]:
import xarray as xr

DEFAULT_SOURCE_PREFIX = "/g/data/fx3/gbr4_2.0"
DEFAULT_PUBLIC_PREFIX = "https://thredds.nci.org.au/thredds/fileServer/fx3/gbr4_v2/"
DEFAULT_LOADABLE_VARS = ["time", "zc", "latitude", "longitude"]
DEFAULT_DROP_VARS = ["botz"]

DEFAULT_PATHS = list(Path(DEFAULT_SOURCE_PREFIX).rglob("*.nc"))

outfile = Path("/scratch/tm70/ct1163/virtualized/virtual_refs")
if outfile.exists():
    raise SystemExit(f"outfile already exists: {outfile}")

paths = [Path(path).expanduser().resolve() for path in DEFAULT_PATHS]
missing = [str(path) for path in paths if not path.exists()]
if missing:
    raise SystemExit(f"source files not found: {missing}")

for path in paths:
    if not str(path).startswith(DEFAULT_SOURCE_PREFIX.rstrip("/") + "/"):
        raise SystemExit(
            f"source file {path} is outside --source-prefix {DEFAULT_SOURCE_PREFIX}"
        )

registry = ObjectStoreRegistry({"file:///": LocalStore(prefix=DEFAULT_SOURCE_PREFIX)})
parser = HDFParser()

vds = open_virtual_mfdataset(
    [path.as_uri() for path in paths],
    registry=registry,
    parser=parser,
    combine="nested",
    concat_dim="time",
    parallel="dask",
    decode_times=False,
    coords="minimal",
    compat="override",
    drop_variables=DEFAULT_DROP_VARS,
    loadable_variables=DEFAULT_LOADABLE_VARS,
)


vds.vz.to_kerchunk(str(outfile), format="parquet")
print(f"wrote {outfile}")


def rewrite_parquet_paths(root: Path, strip_prefix: str, add_prefix: str) -> int:
    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise SystemExit(
            "pyarrow is required for parquet path rewriting; install it or rerun with --no-rewrite-refs"
        ) from exc

    updates = 0
    for parquet_file in root.rglob("*.parq"):
        table = pq.read_table(parquet_file)
        if "path" not in table.column_names:
            continue

        paths = table["path"].to_pylist()
        new_paths = []
        changed = False
        for value in paths:
            if isinstance(value, str) and value.startswith(strip_prefix):
                new_value = add_prefix + value[len(strip_prefix):]
                changed = changed or new_value != value
                new_paths.append(new_value)
            else:
                new_paths.append(value)

        if not changed:
            continue

        columns = []
        for name in table.column_names:
            if name == "path":
                columns.append(pa.array(new_paths))
            else:
                columns.append(table[name])
        pq.write_table(pa.table(columns, names=table.column_names), parquet_file)
        updates += 1

    return updates


updates = rewrite_parquet_paths(outfile, DEFAULT_SOURCE_PREFIX, DEFAULT_PUBLIC_PREFIX)

NotImplementedError: The ManifestArray class cannot concatenate arrays which were stored using different codecs, But found codecs (BytesCodec(endian=<Endian.little: 'little'>), Shuffle(codec_name='numcodecs.shuffle', codec_config={'elementsize': 4}), Zlib(codec_name='numcodecs.zlib', codec_config={'level': 2})) vs (BytesCodec(endian=<Endian.little: 'little'>), Shuffle(codec_name='numcodecs.shuffle', codec_config={'elementsize': 4}), Zlib(codec_name='numcodecs.zlib', codec_config={'level': 3})) .See https://github.com/zarr-developers/zarr-specs/issues/288